# A mixture-of-experts router from scratch
**LLMs From Basics · M5.3 hands-on** &nbsp; [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abhishekaryan23/LLM-Learner/blob/main/course/notebooks/m5-3-moe-router.ipynb)

In the lesson you watched a router send tokens to experts. In NumPy you'll build:

1. A top-k router with softmax gates
2. A tiny MoE layer, checked against a slow reference
3. Expert load and dropped tokens under a capacity limit
4. The Switch Transformer load-balancing loss

Each exercise has a `# TODO`, followed by a check cell with `assert`s. Solutions are at the end. Runs on CPU in seconds.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
np.set_printoptions(precision=3, suppress=True)

T, d, E, d_ff = 32, 16, 8, 32          # tokens, model width, experts, expert hidden size
X = rng.normal(size=(T, d))             # one hidden vector per token
W_r = rng.normal(size=(d, E)) * 0.5     # router weights
experts = [(rng.normal(size=(d, d_ff)) / np.sqrt(d), rng.normal(size=(d_ff, d)) / np.sqrt(d_ff)) for _ in range(E)]

def expert_forward(i, x):
    W1, W2 = experts[i]
    return np.maximum(x @ W1, 0) @ W2   # a small ReLU MLP
print(X.shape, W_r.shape, len(experts))

## 1. Top-k routing with softmax gates

The router turns each token's vector into one probability per expert, keeps the top $k$, and renormalizes them into gates:

$$p = \mathrm{softmax}(x\,W_r), \qquad g_i = \frac{p_i}{\sum_{j \in \mathrm{TopK}(p)} p_j}$$

Write a numerically stable `softmax(z)` (over the last axis) and `route(X, W_r, k)` returning

- `probs`: shape (T, E), the full softmax
- `topk_idx`: shape (T, k), expert indices sorted from highest to lowest probability
- `gates`: shape (T, k), the renormalized probabilities of those experts

In [ ]:
def softmax(z):
    # TODO: subtract the max along the last axis before exponentiating
    raise NotImplementedError

def route(X, W_r, k):
    # TODO: return probs, topk_idx, gates
    raise NotImplementedError

In [ ]:
assert np.allclose(softmax(np.array([1000.0, 1000.0])), [0.5, 0.5]), "softmax must be stable"
probs, topk_idx, gates = route(X, W_r, k=2)
assert probs.shape == (T, E) and topk_idx.shape == (T, 2) and gates.shape == (T, 2)
assert np.allclose(probs.sum(1), 1) and np.allclose(gates.sum(1), 1)
assert (topk_idx[:, 0] == probs.argmax(1)).all()
assert (np.take_along_axis(probs, topk_idx, 1)[:, 0] >= np.take_along_axis(probs, topk_idx, 1)[:, 1]).all()
# the lesson's example: p(E3) = 0.31, p(E6) = 0.19 -> gates 0.62, 0.38
p = np.array([[0.05, 0.08, 0.31, 0.06, 0.07, 0.19, 0.10, 0.14]])
_, idx, g = route(np.log(p), np.eye(8), k=2)
assert idx.tolist() == [[2, 5]] and np.allclose(g, [[0.62, 0.38]])
print("token 0 goes to experts", topk_idx[0], "with gates", gates[0])
print("✓ router works")

## 2. The MoE layer

$$y = \sum_{i \in \mathrm{TopK}} g_i\,E_i(x)$$

Write `moe_forward(X, W_r, k)` that returns the output `Y` (shape T × d) **and** the number of expert evaluations it performed. Process each expert once, on just the tokens routed to it (this is how real implementations batch the work), rather than looping over tokens.

In [ ]:
def moe_forward(X, W_r, k):
    probs, topk_idx, gates = route(X, W_r, k)
    Y = np.zeros_like(X)
    n_evals = 0
    for i in range(E):
        # TODO: find the (token, slot) pairs that chose expert i,
        #       run expert_forward(i, ...) on those tokens once,
        #       add gate * output into the right rows of Y,
        #       and add the number of tokens processed to n_evals
        pass
    return Y, n_evals

In [ ]:
def moe_reference(X, W_r, k):
    probs, topk_idx, gates = route(X, W_r, k)
    Y = np.zeros_like(X)
    for t in range(len(X)):
        for j in range(k):
            Y[t] += gates[t, j] * expert_forward(topk_idx[t, j], X[t:t + 1])[0]
    return Y

for k in (1, 2):
    Y, n = moe_forward(X, W_r, k)
    assert np.allclose(Y, moe_reference(X, W_r, k)), f"mismatch for k={k}"
    assert n == T * k, "each token should be processed by exactly k experts"
dense_evals = T * E
print(f"top-2 MoE ran {T * 2} expert evaluations; running every expert would take {dense_evals}")
print("✓ MoE forward matches the reference")

## 3. Expert load and capacity

Write `expert_load(topk_idx, E)` returning the fraction of **slots** (token-expert pairs) sent to each expert; it sums to 1.

Then write `dropped_slots(topk_idx, E, capacity_factor)`: each expert can take at most $C = \lceil \text{capacity factor} \times T k / E \rceil$ slots; return how many slots overflow in total.

In [ ]:
def expert_load(topk_idx, E):
    # TODO: fraction of all slots that went to each expert, shape (E,)
    raise NotImplementedError

def dropped_slots(topk_idx, E, capacity_factor):
    # TODO
    raise NotImplementedError

In [ ]:
_, topk_idx, _ = route(X, W_r, k=2)
f = expert_load(topk_idx, E)
assert f.shape == (E,) and np.isclose(f.sum(), 1)
# the lesson's unbalanced example: loads [2,1,7,2,1,6,3,2] with capacity ceil(1.25*24/8) = 4 -> 5 dropped
demo = np.repeat(np.arange(8), [2, 1, 7, 2, 1, 6, 3, 2]).reshape(12, 2)   # only the counts matter here
assert dropped_slots(demo, 8, 1.25) == 5
assert dropped_slots(demo, 8, 2.0) == 1    # capacity 6: only E3's 7th slot overflows
assert dropped_slots(demo, 8, 2.5) == 0
print("load per expert:", f)
print("dropped at capacity factor 1.0 / 1.25 / 2.0:", [dropped_slots(topk_idx, E, c) for c in (1.0, 1.25, 2.0)])
print("✓ load and capacity")

## 4. The load-balancing loss

The Switch Transformer (Fedus et al., 2022) adds

$$\mathcal{L}_{\text{aux}} = \alpha \cdot E \sum_{i=1}^{E} f_i\,P_i$$

where $f_i$ is the fraction of slots routed to expert $i$ (your `expert_load`) and $P_i$ is the mean router probability for expert $i$ over the batch. Write `aux_loss(probs, topk_idx, alpha=1.0)`. With perfectly uniform routing it equals $\alpha$.

In [ ]:
def aux_loss(probs, topk_idx, alpha=1.0):
    # TODO
    raise NotImplementedError

In [ ]:
# uniform probabilities and perfectly even assignment -> exactly alpha
uniform_p = np.full((16, 8), 1 / 8)
even_idx = np.arange(16 * 2).reshape(16, 2) % 8
assert np.isclose(aux_loss(uniform_p, even_idx), 1.0)
assert np.isclose(aux_loss(uniform_p, even_idx, alpha=0.01), 0.01)
# a router that loves expert 0 is penalized
W_bad = W_r.copy()
W_bad[:, 0] += 1.5
probs_b, idx_b, _ = route(X, W_bad, 2)
probs_g, idx_g, _ = route(X, W_r, 2)
print(f"aux loss: original router {aux_loss(probs_g, idx_g):.3f}, biased router {aux_loss(probs_b, idx_b):.3f}")
assert aux_loss(probs_b, idx_b) > aux_loss(probs_g, idx_g) > 1.0 - 1e-9
print("✓ the loss grows as routing gets more lopsided")

## Experiment: balancing without an auxiliary loss

DeepSeek-V3 balances its experts with a per-expert **bias** that is added to the routing scores only when choosing the top-k (the gates still use the original probabilities). After each batch, the bias of over-loaded experts goes down and that of under-loaded experts goes up by a small step $\gamma$.

Run the loop below on fresh random batches, starting from the lopsided router, and watch the loads even out. Try different `gamma` values, or `k=1`.

In [ ]:
def route_with_bias(X, W_r, k, bias):
    probs = softmax(X @ W_r)
    topk_idx = np.argsort(-(np.log(probs) + bias), axis=-1)[:, :k]   # bias only affects the choice
    return probs, topk_idx

gamma, k, steps = 0.05, 2, 200
bias = np.zeros(E)
history = []
for step in range(steps):
    Xb = rng.normal(size=(256, d))
    _, idx = route_with_bias(Xb, W_bad, k, bias)
    load = expert_load(idx, E)
    history.append(load)
    bias -= gamma * np.sign(load - 1 / E)
history = np.array(history)

plt.figure(figsize=(7, 4))
plt.plot(history)
plt.axhline(1 / E, color="k", ls="--", lw=1, label="perfect balance")
plt.xlabel("batch")
plt.ylabel("fraction of slots")
plt.title("Per-expert load with bias-based balancing")
plt.legend()
plt.show()
print("first batch:", history[0].round(3))
print("last batch: ", history[-1].round(3))

---
## Solutions
Only run these if you're stuck. They redefine the functions from the exercises.

### Solution 1

In [ ]:
def softmax(z):
    z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)

def route(X, W_r, k):
    probs = softmax(X @ W_r)
    topk_idx = np.argsort(-probs, axis=-1)[:, :k]
    top_p = np.take_along_axis(probs, topk_idx, axis=-1)
    gates = top_p / top_p.sum(axis=-1, keepdims=True)
    return probs, topk_idx, gates

### Solution 2

In [ ]:
def moe_forward(X, W_r, k):
    probs, topk_idx, gates = route(X, W_r, k)
    Y = np.zeros_like(X)
    n_evals = 0
    for i in range(E):
        tok, slot = np.nonzero(topk_idx == i)
        if len(tok) == 0:
            continue
        out = expert_forward(i, X[tok])
        np.add.at(Y, tok, gates[tok, slot][:, None] * out)
        n_evals += len(tok)
    return Y, n_evals

### Solution 3

In [ ]:
def expert_load(topk_idx, E):
    counts = np.bincount(topk_idx.ravel(), minlength=E)
    return counts / topk_idx.size

def dropped_slots(topk_idx, E, capacity_factor):
    counts = np.bincount(topk_idx.ravel(), minlength=E)
    C = int(np.ceil(capacity_factor * topk_idx.size / E))
    return int(np.maximum(counts - C, 0).sum())

### Solution 4

In [ ]:
def aux_loss(probs, topk_idx, alpha=1.0):
    E = probs.shape[1]
    f = expert_load(topk_idx, E)
    P = probs.mean(axis=0)
    return alpha * E * np.sum(f * P)